# Working with outputs: what successive transformations retain

This investigation follows the same A/B/C locations from their degree-one persistence diagram into distances, Betti curves, landscapes, persistence images and scalar summaries. Keep the underlying data fixed until the controlled perturbation is introduced.

All examples use Rips persistence in $H_1$ over $\mathbb F_2$. Replace the `base` array with another $n\times d$ point cloud to send new data through the same diagram, comparison and vectorisation pipeline.

**Working rule.** Before each TODO, predict what information the next representation will retain and what it will forget.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
RNG = np.random.default_rng(5017)
from ripser import ripser
from persim import bottleneck
from scipy.optimize import linear_sum_assignment

def betti_curve(D, grid):
    return np.array([np.sum((D[:, 0] <= a) & (a < D[:, 1])) for a in grid])

def persistence_landscape(D, grid, levels=3):
    tents = np.array([np.maximum(0, np.minimum(grid-b, d-grid)) for b, d in D])
    return np.sort(tents, axis=0)[::-1][:levels]

def persistence_image(D, n=60, sigma=2.8):
    finite = D[np.isfinite(D[:, 1])]
    points = np.c_[finite[:, 0], finite[:, 1]-finite[:, 0]]
    xmax = max(1, points[:, 0].max()*1.08); ymax = max(1, points[:, 1].max()*1.08)
    xs = np.linspace(0, xmax, n); ys = np.linspace(0, ymax, n)
    X, Y = np.meshgrid(xs, ys); image = np.zeros_like(X)
    for birth, persistence in points:
        image += persistence*np.exp(-((X-birth)**2+(Y-persistence)**2)/(2*sigma**2))
    return image, (0, xmax, 0, ymax)

def finite_lengths(D):
    F = D[np.isfinite(D[:, 1])]
    return F[:, 1]-F[:, 0] if len(F) else np.array([])

def max_persistence(D):
    lengths = finite_lengths(D); return float(np.max(lengths)) if len(lengths) else 0.

def total_persistence(D, q=1):
    return float(np.sum(finite_lengths(D)**q))

def persistence_entropy(D):
    lengths = finite_lengths(D); total = lengths.sum()
    if total <= 0: return 0.
    probabilities = lengths/total
    return float(-np.sum(probabilities*np.log(probabilities)))

def wasserstein_linf(D, E, q=1):
    D = D[np.all(np.isfinite(D), axis=1)]; E = E[np.all(np.isfinite(E), axis=1)]
    n, m = len(D), len(E); C = np.full((n+m, n+m), np.inf)
    if n and m: C[:n, :m] = np.max(np.abs(D[:, None, :]-E[None, :, :]), axis=2)**q
    for i, (b, d) in enumerate(D): C[i, m+i] = ((d-b)/2)**q
    for j, (b, d) in enumerate(E): C[n+j, j] = ((d-b)/2)**q
    C[n:, m:] = 0
    rows, cols = linear_sum_assignment(C)
    return float(C[rows, cols].sum()**(1/q))


## Hand checkpoint: one interval in two displays

Take the interval $[18.15,36.08)$. Mark its horizontal extent in a barcode and its point $(18.15,36.08)$ in a persistence diagram.

1. What is its persistence?
2. At which scales does it contribute one to the Betti curve?
3. How far is its diagram point from the diagonal under $L_\infty$?


In [ ]:
birth, death = 18.15, 36.08
print('persistence:', death-birth)
print('distance to diagonal:', (death-birth)/2)


## 1. Observe: reconstruct the guiding A/B/C locations

Each small B-shaped group contains thirteen locations; eleven such groups form the larger A arrangement. This is the same point-location representation used in the reader. It is not a rasterised letter image.


In [ ]:
inner = np.array([
    [0,0], [0,22], [0,44], [0,66], [0,88],
    [22,0], [42,5], [50,23], [22,44], [43,44],
    [50,65], [42,85], [22,88]
], dtype=float)
outer = np.array([
    [350,35,.82], [292,105,.82], [408,105,.82],
    [235,183,.82], [465,183,.82], [182,263,.82],
    [518,263,.82], [300,225,.78], [400,225,.78],
    [125,340,.82], [575,340,.82]
], dtype=float)
base = np.vstack([inner*scale + [x,y] for x,y,scale in outer])
perturbed = base + RNG.normal(0, 1.5, base.shape)
surrogate = base.copy(); surrogate[:, 1] = RNG.permutation(surrogate[:, 1])
datasets = {'A/B/C': base, 'perturbed': perturbed, 'shuffled-y surrogate': surrogate}
diagrams = {name: ripser(points, maxdim=1)['dgms'][1] for name, points in datasets.items()}

fig, axes = plt.subplots(1, 3, figsize=(11, 3.5))
for ax, (name, points) in zip(axes, datasets.items()):
    ax.scatter(*points.T, s=10, color='#267f82')
    ax.set_title(name); ax.set_aspect('equal'); ax.axis('off'); ax.invert_yaxis()
plt.show()

D = diagrams['A/B/C']
finite = D[np.isfinite(D[:, 1])]
fig, axes = plt.subplots(1, 2, figsize=(9, 3.5))
for row, (b, d) in enumerate(sorted(finite, key=lambda pair: pair[0])):
    axes[0].plot([b, d], [row, row], color='#267f82', lw=2)
limit = finite[:, 1].max()*1.06
axes[1].plot([0, limit], [0, limit], '--', color='#819092')
axes[1].scatter(finite[:, 0], finite[:, 1], color='#267f82', s=24)
axes[0].set(xlabel='closeness scale', yticks=[], title='barcode')
axes[1].set(xlabel='birth', ylabel='death', xlim=(0,limit), ylim=(0,limit), title='persistence diagram')
plt.show()


## 2. Predict before transforming the diagram

1. Where should the repeated B-scale intervals appear in the Betti curve?
2. Which summary should retain the later A-scale interval most visibly?
3. Which should change more under many small perturbations: bottleneck or order-one Wasserstein distance?
4. What does independently shuffling one coordinate preserve, and what structure does it destroy?


## 3. Implement

### A. Compare the original and perturbed diagrams

Bottleneck records the worst unavoidable matching cost. The supplied `wasserstein_linf` uses the same $L_\infty$ ground cost and accumulates order-$q$ matching costs. Both allow diagonal matches.

**Checkpoint.** Print the number of finite $H_1$ intervals in each diagram, complete the two calls and confirm that every distance from a diagram to itself is zero.


In [ ]:
for name, diagram in diagrams.items():
    print(name, 'finite H1 intervals:', np.sum(np.isfinite(diagram[:, 1])))

distance_rows = []
for other in ['perturbed', 'shuffled-y surrogate']:
    # TODO: replace None with the appropriate distance calls.
    bottleneck_value = None
    wasserstein_value = None  # use wasserstein_linf(..., q=1)
    distance_rows.append((other, bottleneck_value, wasserstein_value))
print(distance_rows)


### B. Derive several representations from the same intervals

Maximum persistence, total persistence and persistence entropy reduce the diagram to one number. A Betti curve counts intervals alive at each scale. A landscape orders interval tents; a persistence image smooths weighted points onto a fixed grid.

**Checkpoint.** Predict the prominent scales before plotting. Then explain one distinction each representation can no longer recover.


In [ ]:
for name, diagram in diagrams.items():
    # TODO: print maximum persistence, total persistence and persistence entropy.
    pass

grid = np.linspace(0, 100, 220)
base_curve = betti_curve(diagrams['A/B/C'], grid)
landscapes = persistence_landscape(diagrams['A/B/C'], grid, levels=3)
image, extent = persistence_image(diagrams['A/B/C'])
print('five Betti-curve checks:', base_curve[::44])
print('landscape shape:', landscapes.shape, 'image shape:', image.shape)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
# TODO: plot the Betti curve on axes[0].
# TODO: plot each landscape level on axes[1].
# TODO: display the persistence image on axes[2] using extent=extent and origin='lower'.
axes[0].set(xlabel='closeness scale', ylabel='beta_1', title='Betti curve')
axes[1].set(xlabel='closeness scale', ylabel='landscape value', title='landscape')
axes[2].set(xlabel='birth', ylabel='persistence', title='persistence image')
plt.show()


## 4. Compare with a controlled surrogate

Generate shuffled-y surrogates through the complete pipeline. Begin with five repetitions, then use 40. The shuffle preserves each coordinate marginal but destroys their pairing and the visible A/B/C organisation.

Choose maximum persistence as the statistic before seeing the null distribution. This is a teaching randomisation pattern, not automatically a valid scientific test.


In [ ]:
n_surrogates = 5  # checkpoint; change to 40 after the loop works
null_values = []
for repetition in range(n_surrogates):
    shuffled = base.copy()
    # TODO: independently permute the y coordinate.
    # TODO: compute its H1 diagram and append maximum persistence.
    pass

observed = max_persistence(diagrams['A/B/C'])
print('observed:', observed, 'null count:', len(null_values))
# TODO: after setting n_surrogates=40, plot the null histogram and observed line.


## 5. Interpret

1. Which representation best preserves the distinction between the repeated B-scale loops and the later A-scale loop?
2. Which summary emphasises one feature, and which accumulates many differences?
3. What pairing information is lost by the Betti curve?
4. Which choices control the landscape and persistence image?
5. What mechanism would justify the surrogate?
6. Which non-topological baseline should be evaluated beside the selected summary?

The diagram encodes the persistence module; the curve, image and scalar transform or compress that diagram. None is the observed system itself.


## Empirical investigation: a reduced grid-cell state cloud

The course file contains processed population states from one public grid-cell recording. It is deliberately smaller and simpler than the published analysis. Use it to test sensitivity and null design, not to reproduce or independently establish the published torus result.

Work first with a reproducible subsample of the supplied six-dimensional PCA scores. Compare several subsample seeds and dimensions. Then construct a coordinate-wise shuffled surrogate: each PCA-coordinate marginal is retained, but the joint arrangement of coordinates across observations is disrupted.

**Interpretation boundary.** This coordinate shuffle is a teaching null, not the paper's independently time-shifted spike-train null. State exactly what it preserves and destroys. Restricting the exercise to $H_1$ also cannot test the full torus signature, which includes $H_2$.

In [ ]:
from pathlib import Path
from io import BytesIO
from urllib.request import urlopen

def find_course_file(name):
    candidates = [Path(name), Path('data')/name, Path('../../data')/name]
    return next((p for p in candidates if p.exists()), None)

grid_path = find_course_file('grid_cell_torus_r_day1_module2_teaching.npz')
if grid_path is None:
    url = 'https://shannondeealgar.github.io/tda-masterclass/data/grid_cell_torus_r_day1_module2_teaching.npz'
    grid_source = BytesIO(urlopen(url).read())
else:
    grid_source = grid_path

with np.load(grid_source) as subset:
    grid_scores = subset['pca_scores']

sample_rng = np.random.default_rng(6024)
chosen = sample_rng.choice(len(grid_scores), size=280, replace=False)
X = grid_scores[chosen, :4]
X = (X - X.mean(axis=0)) / X.std(axis=0)
X_null = X.copy()
for column in range(X_null.shape[1]):
    sample_rng.shuffle(X_null[:, column])

# TODO: compute the H1 diagrams for X and X_null.
D_grid = np.empty((0, 2))
D_grid_null = np.empty((0, 2))
# TODO: compare bottleneck distance, maximum persistence and Betti curves.
# Repeat for 2, 4 and 6 coordinates and at least three subsample seeds.
# Which findings are stable enough to report, and which remain properties
# of this reduced teaching representation rather than the recorded system?

## Optional extension: what ordinary persistence forgets about labels

The two panels below contain exactly the same point locations. Only the red/blue labels differ. Predict what ordinary Rips persistence will report after the labels are deleted. Then describe a chromatic question that distinguishes mixing from separation.

In [ ]:
angles = np.linspace(0, 2*np.pi, 96, endpoint=False)
radii = np.repeat([0.65, 1.0], 48)
theta = np.tile(angles[:48], 2)
label_cloud = np.c_[radii*np.cos(theta), radii*np.sin(theta)]
mixed = np.arange(96) % 2
separated = (radii > 0.8).astype(int)
fig, axes = plt.subplots(1, 2, figsize=(8, 3.5))
for ax, labels, title in zip(axes, [mixed, separated], ['labels mixed', 'labels separated by ring']):
    ax.scatter(*label_cloud.T, c=np.where(labels, 'tab:blue', 'tab:red'), s=18)
    ax.set_title(title); ax.set_aspect('equal'); ax.axis('off')
plt.show()
# TODO: verify that deleting labels gives the same ordinary H1 diagram in both cases.